# 03_baseline_modeling.ipynb
# Capstone Project: Housing Affordability Stress in High-Growth U.S. Metros
# Author: Shyamsunder Mutcha
# Date: March 2026

In [218]:
import os

# 1. Check for Data (from Notebook 01)
data_file = '../data/cleaned_housing_affordability.csv'
if not os.path.exists(data_file):
    print("Data not found. Running 01_data_ingestion_cleaning.ipynb...")
    %run 01_data_ingestion_cleaning.ipynb
else:
    print("✓ Cleaned data found.")

# 2. Check for Plots (from Notebook 02)
# We check for a specific file (e.g., 04_correlation_heatmap.png)
# instead of just the folder to ensure the plots were actually generated.
expected_plot = '../plots/04_correlation_heatmap.png'
if not os.path.exists(expected_plot):
    print("Plots not found. Running 02_eda_visualizations.ipynb...")
    %run 02_eda_visualizations.ipynb
else:
    print("✓ EDA plots found.")

print("All dependencies verified. Proceeding to Modeling.")

✓ Cleaned data found.
✓ EDA plots found.
All dependencies verified. Proceeding to Modeling.


In [219]:
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import warnings

In [220]:
# --- 2. Data Loading ---
df = pd.read_csv('../data/cleaned_housing_affordability.csv')

# Define features and targets
features = ['Unemployment_Rate', 'Tech_Wage_Median']
X = df[features]
y_reg = df['Price_to_Income_Ratio']
y_clf = (df['Rent_Burden_Pct'] > 30.0).astype(int)

In [221]:
# --- 3. Preprocessing ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [222]:
# --- 4. Linear Regression ---
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_reg, test_size=0.3, random_state=42)
lin_reg = LinearRegression().fit(X_train, y_train)
y_pred_reg = lin_reg.predict(X_test)

print("--- Linear Regression Results ---")
print(f"R²: {r2_score(y_test, y_pred_reg):.3f}")
print(f"MSE: {mean_squared_error(y_test, y_pred_reg):.3f}")
print(f"Coefficients: Tech Wages={lin_reg.coef_[1]:.4f}, Unemployment={lin_reg.coef_[0]:.4f}")

--- Linear Regression Results ---
R²: 0.422
MSE: 1.359
Coefficients: Tech Wages=1.6512, Unemployment=0.1336


In [223]:
# The R² value of 0.422 indicates that our features explain roughly 42% of the variance in housing stress.
# The high coefficient for Tech Wages confirms it is a stronger predictor than Unemployment.

**Rationale for Evaluation Metrics**: **Recall** was prioritized for the classification model to ensure "High Stress" areas are not missed (False Negatives), which is critical for proactive policy planning. **R-squared** was used for regression to quantify the percentage of variance explained by our economic features.

In [224]:
# --- 5. Logistic Regression & Grid Search ---
param_grid = {'C': [0.01, 0.1, 1, 10]}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    grid = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5, scoring='accuracy')
    grid.fit(X_scaled, y_clf)

    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_scaled, y_clf, test_size=0.3, random_state=42)
    final_log_reg = LogisticRegression(C=grid.best_params_['C']).fit(X_train_c, y_train_c)
    y_pred_clf = final_log_reg.predict(X_test_c)

print("\n--- Logistic Regression Results ---")
print(f"Best CV Accuracy: {grid.best_score_:.3f}")
print(f"Test Accuracy: {accuracy_score(y_test_c, y_pred_clf):.3f}")
print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_clf, zero_division=0))


--- Logistic Regression Results ---
Best CV Accuracy: 0.667
Test Accuracy: 0.667

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.33      0.50         3
           1       0.60      1.00      0.75         3

    accuracy                           0.67         6
   macro avg       0.80      0.67      0.62         6
weighted avg       0.80      0.67      0.62         6

